In [26]:
import os
from IPython.display import Markdown, display
import re, json
from datetime import datetime
from openai import OpenAI

In [ ]:
def _clean_transcript(text):
    text = re.sub(r"\[\d{1,2}:\d{2}(:\d{2})?\]", "", text)   # remove timestamps
    text = re.sub(r"\b[A-Z][a-z0-9_]{0,20}:\s+", "", text)   # naive speaker labels
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _format_owner(o):
    if isinstance(o, list):
        if len(o) == 0:
            return "TBD"
        if len(o) == 1:
            return str(o[0])
        if len(o) == 2:
            return f"{o[0]} & {o[1]}"
        return ", ".join(map(str, o[:-1])) + " & " + str(o[-1])
    return str(o)

def _normalize_result(obj):
    out = {"title": "", "summary": "", "highlights": [], "actions": []}
    if not isinstance(obj, dict):
        out["summary"] = str(obj)
        return out

    out["title"] = obj.get("title", "").strip()
    out["summary"] = obj.get("summary", "").strip()
    out["highlights"] = obj.get("highlights", []) if isinstance(obj.get("highlights", []), list) else []
    raw_actions = obj.get("actions", []) if isinstance(obj.get("actions", []), list) else []
    norm_actions = []
    for a in raw_actions:
        if isinstance(a, dict):
            owner = a.get("owner", "TBD")
            if owner is None:
                owner = "TBD"
            owner = _format_owner(owner)
            norm_actions.append({
                "task": (a.get("task") or "").strip(),
                "owner": owner,
                "due": a.get("due", "") or "",
                "notes": a.get("notes", "") or ""
            })
        else:
            norm_actions.append({"task": str(a), "owner": "TBD", "due": "", "notes": ""})
    out["actions"] = norm_actions
    return out

def summarize_meeting(transcript_path,):
    # Load and clean the transcript
    txt = open(transcript_path, "r", encoding="utf-8").read()
    clean = _clean_transcript(txt)

    # Prompt the LLM to extract a concise summary and action items
    system_prompt = (
        "You are a friendly, professional meeting assistant who writes like a helpful teammate. "
        "Return ONLY valid JSON with this schema: "
        '{"title":"...","summary":"...", "highlights":[{"point":"..."}], "actions":[{"task":"...","owner":"...","due":"...","notes":"..."}]}. '
        "Write `title` as a short, descriptive header (4–8 words) that someone could use as the email subject or section heading. "
        "Write `summary` as a natural, human-readable paragraph (3–5 sentences) suitable to paste into an email: include key decisions, current status, and next steps in clear, conversational language. "
        "Provide `highlights` as 2–4 short, concise bullet-style points. "
        "List `actions` as a cohesive, friendly array of items where each `task` begins with a verb (e.g., 'Finalize acceptance criteria'), `owner` is a person or team name or 'TBD', `due` is an ISO date (YYYY-MM-DD) or an empty string, and `notes` is one short sentence providing helpful context. "
        "Keep actions concise, avoid duplicates, and merge closely related sub-tasks into a single action when appropriate. "
        "If you are unsure about an owner or due date, use 'TBD' or an empty string—do not invent details. "
        "Do not include any text outside the JSON object. Use proper grammar, be helpful, and prioritize clarity and a collaborative tone."
    )
    user_prompt = f"Here is the clean transcript:\n\n{clean}\n\nReturn only the JSON described above. If no actions, use an empty array for \"actions\"."
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    resp = client.chat.completions.create(model="llama3.2", messages=messages)
    raw = resp.choices[0].message.content

    print("Raw LLM output:", raw)
    try:
        return _normalize_result(json.loads(raw))
    except Exception:
        m = re.search(r"\{.*\}", raw, re.S)
        if m:
            return _normalize_result(json.loads(m.group(0)))
        return _normalize_result({"summary": raw.strip(), "actions": []})


In [ ]:
def _display_result(result):
    title = result.get("title", "").strip()
    summary = result.get("summary", "").strip()
    highlights = result.get("highlights") or []
    actions = result.get("actions") or []

    if title:
        display(Markdown(f"### {title}\n"))
    display(Markdown("**Summary**\n\n" + (summary or "(no summary)")))

    if highlights:
        h_lines = []
        for h in highlights:
            if isinstance(h, dict):
                point = h.get("point", "")
            else:
                point = str(h)
            if point:
                h_lines.append(f"- {point}")
        if h_lines:
            display(Markdown("**Highlights**\n" + "\n".join(h_lines)))

    if actions:
        lines = []
        for a in actions:
            task = a.get("task", "(no task)")
            owner = a.get("owner", "TBD")
            due = a.get("due", "")
            notes = a.get("notes", "")
            line = f"- **{task}** — owner: {owner}"
            if due:
                line += f" — due: {due}"
            if notes:
                line += f"\n  - notes: {notes}"
            lines.append(line)
        display(Markdown("**Action Items**\n" + "\n".join(lines)))
    else:
        display(Markdown("**Action Items**\n- None found"))

# Usage
result = summarize_meeting("meeting_transcript.txt")
_display_result(result)

Raw LLM output: ```
{
  "title": "Project Status Updates and Next Steps",
  "summary": "We review current project status, discuss risks and blockers, and outline next steps. The product team prioritizes ingestion hardening, model validation, and dashboard resilience. We also address infra and deployment, QA and staging readiness, documentation and runbook items, and action items.",
  "highlights": [
    "Resilient ingestion with added sequence IDs for producer and consumer in the short term",
    "Fine-tuning and performance testing on staging, followed by model validation",
    "Dashboard resilience measures to be implemented post rollout"
  ],
  "actions": [
    {
      "task": "Prototype sequence ID ordering in the ingestion prototype and add consumer reorder buffer; push branch and PR with tests",
      "owner": "Alex",
      "due": "2026-05-14",
      "notes": "Test harness must be small and backward-compatible"
    },
    {
      "task": "Request GPU quota increase, post support 

**Summary**

We review current project status, discuss risks and blockers, and outline next steps. The product team prioritizes ingestion hardening, model validation, and dashboard resilience. We also address infra and deployment, QA and staging readiness, documentation and runbook items, and action items.

**Action Items**
- **Prototype sequence ID ordering in the ingestion prototype and add consumer reorder buffer; push branch and PR with tests** — owner: Alex — due: 2026-05-14
- **Request GPU quota increase, post support ticket ID, and confirm staging availability for model validation** — owner: Jordan — due: 2026-05-10
- **Validate fine-tuned model on staging, run evaluation notebooks, and report metrics + any data drift observations** — owner: Priya — due: 2026-05-15
- **Make dashboard resilient to schema changes; implement panel fallbacks and nightly metric-shape comparison job** — owner: Sam — due: 2026-05-21
- **Run automated security scanner on new connector code and provide PII redaction regex patterns; create tickets for any findings** — owner: Carlos — due: 2026-05-11
- **Prepare QA regression plan and schedule test runs (assign two engineers) for ingestion + model validation scenarios** — owner: Nia — due: 2026-05-12
- **Draft stakeholder communication summarizing risks, timeline, and mitigation plan for potential rollout delays** — owner: Mia — due: 2026-05-13
- **Add deterministic seeds to test harness and documentation for reproducible tests** — owner: Alex — due: 2026-05-12
- **Publish infra runbook and perform internal review (Jordan to publish, Mia to review)** — owner: ['Jordan', 'Mia'] — due: 2026-05-11
- **Prepare a short internal demo of dashboard fallbacks for next Wednesday; coordinate demo content with product** — owner: Sam — due: 2026-05-13